# BMY drum transcriber: training notebook

Trains BMY's own drum transcriber (9 drums: kick, snare, 3 toms, closed and open hi-hat, crash, ride) from **licence-clean** material only:

* **E-GMD** drum recordings (Google, CC BY 4.0): 43 kits, read straight out of Google's archive, a file at a time (no 90 GB download).
* Some performances re-played through **BMY's GeneralUser GS kits**.
* **Backing instruments generated here** (bass, keys, guitars, pads, leads), mixed under the drums, then split again with **Demucs** (the separator Groove Lab uses), so the model learns real separation artefacts.

**How to run:** `Runtime › Change runtime type › T4 GPU` (or better), then `Runtime › Run all`. Everything is saved in your Google Drive in **BMY_Transcriber**, so if Colab stops (time limit, closed tab), just **Run all** again and it carries on where it stopped.

**When it finishes**, the last cell shows the scores and saves `bmy_drums.onnx` and `bmy_drums.json` in that Drive folder. Send both to Claude, and they become a third Transcriber option in Groove Lab, next to ADTOF and Magenta.

## 1. Settings
`PRESET = 'trial'` first (about 2 hours in total) to check everything works; then `'full'`.

In [ ]:
PRESET = 'trial'          # 'trial' or 'full'

SETTINGS = {
    'trial': dict(train_hours=10, val_hours=1, test_hours=1, steps=15000),
    'full':  dict(train_hours=40, val_hours=2, test_hours=2, steps=40000),
}[PRESET]
SESSION_HOURS = 3.5       # each step stops after this long and saves; Run all again to continue
WORK = '/content/drive/MyDrive/BMY_Transcriber/' + PRESET
print(PRESET, SETTINGS, '->', WORK)

## 2. Set up (Google Drive, tools, BMY code, SoundFont, E-GMD index)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys, urllib.request
os.makedirs(WORK, exist_ok=True)
subprocess.run('apt-get -qq install -y fluidsynth > /dev/null', shell=True, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'demucs', 'remotezip', 'pretty_midi', 'soundfile', 'onnx', 'onnxruntime'], check=True)
if not os.path.exists('/content/bmy-groove-lab'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/taddmlh/bmy-groove-lab.git', '/content/bmy-groove-lab'], check=True)
else:   # always take the latest code (shallow copy: fetch + reset, not pull)
    subprocess.run('git -C /content/bmy-groove-lab fetch -q --depth 1 origin main && git -C /content/bmy-groove-lab reset -q --hard FETCH_HEAD', shell=True, check=True)
sys.path.insert(0, '/content/bmy-groove-lab/training')
import importlib, bmy_transcriber as bt; importlib.reload(bt)
SF2 = '/content/drive/MyDrive/BMY_Transcriber/GeneralUser-GS.sf2'
if not os.path.exists(SF2):
    urllib.request.urlretrieve(bt.GS_SF2_URL, SF2)
CSV = '/content/drive/MyDrive/BMY_Transcriber/e-gmd-v1.0.0.csv'
if not os.path.exists(CSV):
    urllib.request.urlretrieve(bt.EGMD_CSV, CSV)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none (switch the runtime to a GPU!)')
egmd = bt.EGMD(CSV)
print('E-GMD clips:', len(egmd.rows), '| SoundFont MB:', os.path.getsize(SF2) // 2**20)

## 3. Quick check (a few seconds of each step on one clip)

In [ ]:
import numpy as np, time
r = egmd.select('train', 0.01, seed=99)[0]
t = time.time()
hits = bt.midi_hits(egmd.read(r['midi_filename'])); y = bt.read_wav(egmd.read(r['audio_filename']))
print(f"clip {r['id']} ({r['kit_name']}, {float(r['duration']):.1f} s): {len(hits)} hits, read in {time.time() - t:.1f} s")
t = time.time(); back = bt.render_backing(10, float(r['bpm']), 1, SF2); print(f'backing rendered in {time.time() - t:.1f} s, rms {np.sqrt(np.mean(back**2)):.3f}')
t = time.time(); stem = bt.DEMUCS(bt.normalise(y[:bt.SR * 10] + back[:bt.SR * 10] * 0.5), DEVICE); print(f'Demucs on 10 s: {time.time() - t:.1f} s')
x = bt.features(bt.normalise(stem)); print('features', x.shape, 'targets', bt.targets(hits, len(x)).shape)

## 4. Prepare the data
The longest step (trial: about 1 hour, full: about 4–5 hours, over two sessions). Safe to stop and re-run: finished clips are kept.

In [ ]:
cfg = bt.PrepConfig(work=WORK, sf2=SF2, train_hours=SETTINGS['train_hours'], val_hours=SETTINGS['val_hours'], test_hours=SETTINGS['test_hours'])
budget = SESSION_HOURS * 3600
t0 = time.time()
for split in ('validation', 'test', 'train'):
    left = budget - (time.time() - t0)
    if left < 300:
        print('Session time used up: Run all again to continue.'); break
    bt.prepare(split, cfg, egmd, device=DEVICE, limit_s=left)

## 5. Train
Trial: about 30–45 minutes on a T4. Saves a checkpoint every 2000 steps; re-running continues.

In [ ]:
tcfg = bt.TrainConfig(work=WORK, steps=SETTINGS['steps'])
bt.train(tcfg, device=DEVICE, time_limit_s=SESSION_HOURS * 3600)

## 6. Scores and export
Tunes a threshold per drum on the validation set, scores the held-out test set, and writes `bmy_drums.onnx` + `bmy_drums.json` to the Drive folder.

In [ ]:
import json
meta = bt.export(WORK)
print()
print(f"{'drum':10s} {'F1':>6s} {'precision':>10s} {'recall':>8s} {'hits':>6s}")
for k, v in meta.get('test', {}).items():
    print(f"{k:10s} {v['f1']:6.3f} {v['precision']:10.3f} {v['recall']:8.3f} {v.get('n', ''):>6}")
print('\nFiles in', WORK, ':', [f for f in os.listdir(WORK) if f.startswith('bmy_drums')])